In [8]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import pytz
import numpy as np
#For loading the data

apps_df = pd.read_csv("Play_Store_Data.csv")
reviews_df = pd.read_csv("User_Reviews (1).csv")

#DATA CLEANING

# For converting rating and reviews to numeric:
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'],errors ='coerce')


apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors ='coerce')

# For converting the Install columns by removing comas and + 
apps_df ['Installs'] = apps_df ['Installs'].astype(str)
apps_df ['Installs'] = apps_df ['Installs'].str.replace('+','',regex=False)
apps_df ['Installs'] = apps_df ['Installs'].str.replace(',','',regex=False)
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors ='coerce')

#Now clean price column:
apps_df['Price'] = apps_df ['Price'].astype(str)
apps_df['Price'] = apps_df['Price'].str.replace('$', '', regex=False)
apps_df['Price'] = pd.to_numeric(apps_df['Price'], errors='coerce')

#Now converting the size to MB:

def convert_size(size):
    if isinstance(size, str):
       if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    else:
        return np.nan
apps_df['Size'] = apps_df['Size'].apply(convert_size)

#Now converting last updated to datetime
apps_df['Last Updated'] = pd. to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df['Update_Month'] = apps_df['Last Updated'].dt.month


# TASK 1: Grouped Bar Chart


#Apply Filters

filtered_df = apps_df[
    (apps_df ['Rating'] >= 4.0) &
    (apps_df['Size'] >= 10) &
     (apps_df['Update_Month'] ==1)
]


# Now top 10 categories by Installs:

top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .nlargest(10)
    .index
)

final_df = filtered_df[filtered_df['Category'].isin(top_categories)]

#Now aggregate the data:

summary_df = final_df.groupby('Category').agg(
    Avg_Rating=('Rating', 'mean'),
    Total_Reviews=('Reviews', 'sum')
).reset_index()

# Time Condition from 3 PM to 5 PM IST
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("15:00", "%H:%M").time()
end_time = datetime.strptime("17:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = go.Figure()

    fig.add_bar(
        x=summary_df['Category'],
        y=summary_df['Avg_Rating'],
        name='Average Rating'
    )

    fig.add_bar(
        x=summary_df['Category'],
        y=summary_df['Total_Reviews'],
        name='Total Reviews'
    )

    fig.update_layout(
        title="Average Rating vs Total Reviews (Top 10 Categories)",
        xaxis_title="Category",
        yaxis_title="Value",
        barmode='group',
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
    )

    fig.show()

else:
    print("The Task 1 graph is visible only between 3 PM and 5 PM IST.")

#TASK 2: Chorolopeth Map
                       
import plotly.express as px

# Step 1: Filter categories (not starting with A, C, G, S)
task2_df = apps_df[
    ~apps_df['Category'].str.startswith(('A', 'C', 'G', 'S'), na=False)
].copy()

# Step 2: Calculate total installs per category
category_installs = (
    task2_df
    .groupby('Category', as_index=False)['Installs']
    .sum()
)

# Step 3: Select top 5 categories
top5_df = category_installs.sort_values(
    by='Installs',
    ascending=False
).head(5)

# Step 4: Highlight categories above 1M installs
top5_df['Install_Level'] = np.where(
    top5_df['Installs'] > 1_000_000,
    'Above 1M',
    'Below 1M'
)

# Step 5: Assign dummy country codes (for visualization only)
country_codes = ['USA', 'IND', 'BRA', 'DEU', 'AUS']
top5_df['Country_Code'] = country_codes[:len(top5_df)]

# Step 6: Time condition (6 PM – 8 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("20:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = px.choropleth(
        top5_df,
        locations='Country_Code',
        locationmode='ISO-3',
        color='Installs',
        hover_name='Category',
        hover_data=['Install_Level'],
        title='Global Installs by Top 5 App Categories',
        color_continuous_scale='Plasma'
    )

    fig.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    fig.show()

else:
    print("The Task 2 graph is visible only between 6 PM and 8 PM IST.")


# TASK 3: Dual Axis Chart

# Convert Android version to numeric 
apps_df['Android Ver'] = apps_df['Android Ver'].astype(str)
apps_df['Android Ver'] = apps_df['Android Ver'].str.extract(r'(\d+\.\d+)')
apps_df['Android Ver'] = pd.to_numeric(apps_df['Android Ver'], errors='coerce')

# Create Revenue column
apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

# Apply filters
filtered_df = apps_df[
    (apps_df['Installs'] >= 10000) &
    (apps_df['Revenue'] >= 10000) &
    (apps_df['Android Ver'] > 4.0) &
    (apps_df['Size'] > 15) &
    (apps_df['Content Rating'] == 'Everyone') &
    (apps_df['App'].str.len() <= 30)
].copy()


# Top 3 categories by installs
top_categories = (
    filtered_df
    .groupby('Category')['Installs']
    .sum()
    .nlargest(3)
    .index
)

final_df = filtered_df[filtered_df['Category'].isin(top_categories)].copy()

# Group by Type (Free vs Paid)
summary_df = (
    final_df
    .groupby('Type')
    .agg(
        Avg_Installs=('Installs', 'mean'),
        Avg_Revenue=('Revenue', 'mean')
    )
    .reset_index()
)

# Debug check 
print(summary_df)
print(summary_df.columns)

# Time condition (1 PM to 2 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("13:00", "%H:%M").time()
end_time = datetime.strptime("14:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = go.Figure()

    fig.add_bar(
        x=summary_df['Type'],
        y=summary_df['Avg_Installs'],
        name='Average Installs'
    )

    fig.add_scatter(
        x=summary_df['Type'],
        y=summary_df['Avg_Revenue'],
        name='Average Revenue',
        yaxis='y2',
        mode='lines+markers'
    )

    fig.update_layout(
        title='Average Installs vs Revenue (Free vs Paid Apps)',
        xaxis_title='App Type',
        yaxis=dict(title='Average Installs'),
        yaxis2=dict(
            title='Average Revenue',
            overlaying='y',
            side='right'
        ),
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    fig.show()

else:
    print("Task 3 graph is visible only between 1 PM and 2 PM IST.")


#TASK 4: Time Series Analysis of App Installs (Category Wise): 

task4_df = apps_df.copy()

#Apply all Filters:
task4_df = task4_df[
    (~task4_df['App'].str.lower().str.startswith(('x','y','z'))) &
    (~task4_df['App'].str.contains('S', case=False)) &
    (task4_df['Reviews'] > 500) &
    (task4_df['Category'].str.startswith(('E','C','B')))
]

#Category Translation
category_translation = {
    'Beauty': 'सौंदर्य',
    'Business': 'வணிகம்',
    'Dating': 'Dating'
}

task4_df.loc[:, 'Category_Display'] = (
    task4_df['Category'].replace(category_translation)
)

#Month Creation (AFTER filtering)
task4_df.loc[:, 'Month'] = (
    pd.to_datetime(task4_df['Last Updated'], errors='coerce')
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Aggregation 
monthly_installs = (
    task4_df
    .groupby(['Month', 'Category_Display'], as_index=False)['Installs']
    .sum()
)

#Calculate Month over Month Growth
monthly_installs['MoM_Growth'] = (
    monthly_installs
    .groupby('Category_Display')['Installs']
    .pct_change()
)
# Now setting time condition between 6pm to 9pm:
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("21:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig = go.Figure()

    for category in monthly_installs['Category_Display'].unique():
        category_data = monthly_installs[
            monthly_installs['Category_Display'] == category
        ]

        fig.add_trace(go.Scatter(
            x=category_data['Month'],
            y=category_data['Installs'],
            mode='lines+markers',
            name=category
        ))


    fig.update_layout(
        title="Total Installs Trend Over Time (Category Wise)",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    fig.show()
else:
    print("Task 4 graph is visible only between 6 PM and 9 PM IST.")

#Task- 5 Bubble Chart - App Size vs Average Rating

#Merge apps data with reviews data
task5_df = pd.merge(apps_df, reviews_df, on='App', how='inner')

# Apply all required filters:

task5_df = task5_df[
    (task5_df['Rating'] > 3.5) &
    (task5_df['Reviews'] > 500) &
    (task5_df['Installs'] > 50000) &
    (~task5_df['App'].str.contains('S', case=False, na=False)) &
    (task5_df['Sentiment_Subjectivity'] > 0.5) &
    (task5_df['Category'].isin([
        'GAME', 'BEAUTY', 'BUSINESS', 'COMICS',
        'COMMUNICATION', 'DATING', 'ENTERTAINMENT',
        'SOCIAL', 'EVENTS'
    ]))
]


# Step 3: Translate selected categories for graph display
category_translation = {
    'BEAUTY': 'सौंदर्य',       # Hindi
    'BUSINESS': 'வணிகம்',      # Tamil
    'DATING': 'Dating (Deutsch)'
}

task5_df['Category_Display'] = (
    task5_df['Category'].replace(category_translation)
)


# Now setting time condition between 5pm to 7 pm

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = px.scatter(
        filtered_df,
        x='Size',
        y='Rating',
        size='Installs',
        color='Category_Display',
        title='App Size vs Rating (Bubble Chart)',
        labels={
            'Size':'App Size (MB)',
            'Rating':'Average Rating'
        }
    )

    # Highlight Game category in Pink
    fig.for_each_trace(
        lambda t: t.update(marker_color='pink')
        if 'Game' in t.name else None
    )

    fig.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    fig.show()

else:
    print("Task 5 graph is visible only between 5 PM and 7 PM IST.")


   #Task- 6 : Stacked Area Chart- Cumulative Installs Over Time:

    task6_df = apps_df.copy()
    #Apply all required filters:
    task6_df = task6_df[
    (task6_df['Rating'] >= 4.2) &
    (~task6_df['App'].str.contains(r'\d', regex=True)) &
    (task6_df['Category'].str.startswith(('T', 'P'))) &
    (task6_df['Reviews'] > 1000) &
    (task6_df['Size'].between(20, 80))
]
    #Translate category names:
    category_translation = {
    'Travel & Local': 'Voyage & Local',
    'Productivity': 'Productividad',
    'Photography': '写真'
}

task6_df['Category_Display'] = task6_df['Category'].replace(category_translation)

#Convert last updated into monthly format:
task6_df['Month'] = (
    pd.to_datetime(task6_df['Last Updated'], errors='coerce')
    .dt.to_period('M')
    .dt.to_timestamp()
)
#Aggregation
monthly_installs = (
    task6_df
    .groupby(['Month', 'Category_Display'], as_index=False)['Installs']
    .sum()
)

monthly_installs['Cumulative_Installs'] = (
    monthly_installs
    .groupby('Category_Display')['Installs']
    .cumsum()
)

#Calculate Month-over-Month growth
monthly_installs['MoM_Growth'] = (
    monthly_installs
    .groupby('Category_Display')['Installs']
    .pct_change()
)
# Now setting time condition between 4pm to 6pm

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime("16:00", "%H:%M").time()
end_time   = datetime.strptime("18:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = go.Figure()

    for category in monthly_installs['Category_Display'].unique():

        category_data = monthly_installs[
            monthly_installs['Category_Display'] == category
        ]

        fig.add_trace(go.Scatter(
            x=category_data['Month'],
            y=category_data['Cumulative_Installs'],
            stackgroup='one',
            name=category,
            opacity=0.9
        ))

    fig.update_layout(
        title="Cumulative App Installs Over Time (Category Wise)",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    fig.show()

else:
    print("Task 6 graph is visible only between 4 PM and 6 PM IST.")

The Task 1 graph is visible only between 3 PM and 5 PM IST.
The Task 2 graph is visible only between 6 PM and 8 PM IST.
   Type   Avg_Installs   Avg_Revenue
0  Paid  590769.230769  1.965246e+06
Index(['Type', 'Avg_Installs', 'Avg_Revenue'], dtype='object')
Task 3 graph is visible only between 1 PM and 2 PM IST.
Task 4 graph is visible only between 6 PM and 9 PM IST.
Task 5 graph is visible only between 5 PM and 7 PM IST.
Task 6 graph is visible only between 4 PM and 6 PM IST.
